# Labhub demo

1. Download and install LabHub
  - https://svn.isibrno.cz/levifot/labhub
  - create new virtual environment
  - install dependencies requirements.txt
  - for now manually with my help
  - eventually simple how-to installation guide

2. Specify connected devices 
  - for now simple config.yaml file (`labhub\hub\config.yaml`)
  - ideally 1 config per PC (all connected devices)
  - some devices require further installation (e.g. specific DLL/SKD libraries)
  - eventually nice guide/wizard for each device

3. Start LabHub
  - for now via commandline (`python \hub\tray_app\tray.py`)
    - make sure you are in labhub venv: `View->Command pallete->Select Python Interpreter: (.venv)`
  - this starts a tray application that can:
    - start/stop the server
    - modify/reload config.yaml
    - launch web-based GUI
  - eventually nice labhub.exe file with all dependencies packed together 

4. Browse devices in web GUI
  - list of available devices
  - each device has:
    - properties (e.g. "voltage" or "temperature") - read-out/set
    - commands (e.g. "home stage" or "pump") - initialize an action
    - data (e.g. "time series" or "PSD") - provides bulk data
  - main use-case:
    - tinker with different settings
    - see immediate feedback
    - as reference for scripting
  - eventually more custom GUI/device

5. Scripting
  - any language (for now python, julia in development, matlab possible)
  - all values are synchronized with web GUI (and other clients)
  - simple module (1 file in python), minimal requirements
  - see demo script below ↓↓


## Scripting demo

### 1. connect to labhub server

In [1]:
import client.python as lhc

lhc.connect("127.0.0.1", 8212)
print(lhc)

trying to __init__ Hub with base_url http://127.0.0.1:8212
<module 'client.python' from 'c:\\Users\\jankl\\Documents\\pracovni\\upt\\labglue\\labhub\\client\\python\\__init__.py'>
labhub (127.0.0.1:8212) has the following devices:
  .dummy_01           Minimal example driver.
    Exposes one RW property 'foo' and one command 'bar()'.
  .example_device_01  Spec-driven toy device that generates a time-series from a configurable waveform.
    Properties control the waveform and streaming cadence; commands manage streaming.
  .picoscope          PicoScope 5000a series driver.
    This driver requires the PicoSDK to be installed.
  .kcube              KCube piezo driver
    https://www.thorlabs.com/thorcat/ETN/ETN017657-D03.pdf


###  2. browse devices and their properties/methods


In [2]:
print(lhc.example_device_01)

example_device_01  [example_device]
Spec-driven toy device that generates a time-series from a configurable waveform.
    Properties control the waveform and streaming cadence; commands manage streaming.

Parameters:
  .time_step: float = 0.1  # Sampling interval for generated data (seconds)
  .number_of_time_steps: int = 1000  # How many samples to return from get_timestamps() (default = 1000)
  .noise: bool = True  # Add small uniform noise (~±5% of amplitude).
  .wave_type: str = 'sin'  # Waveform type

Commands:
  .get_timestamps()
    Return x-axis timestamps based on number_of_time_steps and time_step.


### 3. set/get property values

In [4]:
print(lhc.example_device_01.number_of_time_steps)
lhc.example_device_01.number_of_time_steps = 4000
print(lhc.example_device_01.number_of_time_steps)


2568
4000


### 4. call methods

In [5]:
help(lhc.example_device_01.get_timestamps)
ts = lhc.example_device_01.get_timestamps()
print(ts)

Help on function get_timestamps in module client.python.client:

get_timestamps(**kwargs)
    get_timestamps()

    Return x-axis timestamps based on number_of_time_steps and time_step.

[0.0, 0.1, 0.2, 0.30000000000000004, 0.4, 0.5, 0.6000000000000001, 0.7000000000000001, 0.8, 0.9, 1.0, 1.1, 1.2000000000000002, 1.3, 1.4000000000000001, 1.5, 1.6, 1.7000000000000002, 1.8, 1.9000000000000001, 2.0, 2.1, 2.2, 2.3000000000000003, 2.4000000000000004, 2.5, 2.6, 2.7, 2.8000000000000003, 2.9000000000000004, 3.0, 3.1, 3.2, 3.3000000000000003, 3.4000000000000004, 3.5, 3.6, 3.7, 3.8000000000000003, 3.9000000000000004, 4.0, 4.1000000000000005, 4.2, 4.3, 4.4, 4.5, 4.6000000000000005, 4.7, 4.800000000000001, 4.9, 5.0, 5.1000000000000005, 5.2, 5.300000000000001, 5.4, 5.5, 5.6000000000000005, 5.7, 5.800000000000001, 5.9, 6.0, 6.1000000000000005, 6.2, 6.300000000000001, 6.4, 6.5, 6.6000000000000005, 6.7, 6.800000000000001, 6.9, 7.0, 7.1000000000000005, 7.2, 7.300000000000001, 7.4, 7.5, 7.600000000000000

### 5. acquire data

In [9]:
# in development :-)